# L4 — Configuring Datadog AI Guard (Console Tour)

The AI Guard security layer is driven from the **Datadog AI Guard console** — policies, sensitivity, tool
blocklists, detection rules, and the Investigate view where verdicts land. This notebook is a guided tour of every
control, with the one setting you must get right called out: the **service policy must be set to Block** for
in-runtime blocking to fire.

> This is a console tour — there's no code to run. Follow the navigation paths and screenshots to configure each control.

## Prerequisites

- A Datadog org with **AI Guard** access (Limited Availability — confirm it's enabled for your org).
- The agents deployed with `AI_GUARD_ENABLED=true` and `DD_SERVICE` set (notebook 2), so a service exists to police.
- The agents running on the **Datadog trace-agent sidecar image** (L3 Notebooks 2-3 (sidecar deploy)).
  This is what delivers the `ai_guard` verdict span to Datadog — on the default ADOT image the span is dropped
  (ddtrace ↔ `opentelemetry-instrument` collision) and **Investigate stays empty even though enforcement works**.
- A retention filter `resource_name:ai_guard` @ **100%** so evaluations are retained and visible in Investigate.

## What is AI Guard?

Datadog **AI Guard** is an LLM-as-judge that evaluates an agent's conversation — prompts, model outputs, and tool
calls — and returns a verdict (`ALLOW` / `DENY` / `ABORT`) with attack-category tags. In this workshop it runs as a
custom Strands hook inside the agents (notebook 2) and its `ABORT` verdict drives the gateway circuit breaker (notebook 3).
Everything *behavioural* — what to block, how sensitive, which tools — is configured here in the console, **without
redeploying the agents**. That is the pluggability story: change the policy, not the code.

## License Details

Refer to the LICENSE file in the main folder for license details.

## How to Access the Console

**Navigation:** Datadog → **Security** → **AI Guard**. The left nav exposes **Discover**, **Signals /
Investigate**, and **Settings** (Service Policies, Evaluation Sensitivity, Tool Blocklist,
Detection Rules).

---
## 1. Service Policies — Monitor vs. Block  ⭐ (the one required setting)

The single most important setting. A **service policy** sets, per service (and per env), whether AI Guard runs in
**Monitor** (observe only — `evaluate` never raises) or **Block** (enforce — `evaluate` raises `AIGuardAbortError`
on DENY/ABORT). The server policy is a **hard gate**: even with `block=True` in code, a Monitor-mode policy will
never block. **Set the workshop service to Block** for Scenarios 2–4 of the enforcement lab to actually block.

**What to look for:**
- The policy list keyed by **service** / **env** / **service+env** (no per-tool or per-checkpoint mode — that's all there is).
- The **Monitor ↔ Block** toggle for `anycompany-order-agent` / `anycompany-refund-agent` (your `DD_SERVICE` values).
- The sensitive-data toggle on the policy.

**Navigation:** Datadog → Security → AI Guard → Settings → **Service Policies**

![AI Guard service policy — Monitor vs Block](aiguard_service_policy.png)

> **Tip:** After adding the `anycompany-refund-agent` service policy, you may need to
> adjust the **Evaluation sensitivity** slider for it to appear as "Protected" in the
> Discover view. Save the policy after adjusting — the Discover page updates within a minute.


---
## 2. Evaluation Sensitivity

A 0–1 slider (default **0.5**) controlling how aggressively AI Guard flags borderline content. Higher = more
blocks (and more false positives); lower = fewer. Tune per service.

**What to look for:**
- The sensitivity slider and its current value for the workshop service.
- How a change shifts borderline scenarios between ALLOW and DENY (test in the Service Policies).

**Navigation:** Datadog → Security → AI Guard → Settings → **Evaluation Sensitivity**

![AI Guard evaluation sensitivity slider](aiguard_sensitivity.png)

---
## 3. Tool Blocklist

An independent allow/deny list of **named tools**, per service/env. Distinct from the monitor/block posture: it
hard-blocks specific tools regardless of the semantic verdict — useful for high-blast-radius tools.

**What to look for:**
- Adding a tool (e.g. `refund-tools___process_refund`) to the blocklist and seeing the call denied.
- That this is orthogonal to the AI Guard verdict (a blocklisted tool is denied even on an ALLOW verdict).

**Navigation:** Datadog → Security → AI Guard → Settings → **Tool Blocklist**

![AI Guard tool blocklist](aiguard_tool_blocklist.png)


---
## 4. Detection Rules

Detection rules turn AI Guard evals into **Security Signals** and can trigger **automated responses** (e.g. notify,
or block an IP/user). These are **threshold aggregations** ("N DENYs in M minutes") — great for alerting and
forensics, but they lag by seconds-to-minutes and **cannot gate the next tool call**. That is why inline gating here relies on the in-runtime hook + gateway breaker (notebooks 2-3) rather than a detection-rule-driven response.

**What to look for:**
- The default AI Guard detection rules and their signal severity.
- The automated-response options (useful for alerting, but they lag and can't gate the next tool call).

**Navigation:** Datadog → Security → AI Guard → Settings → **Detection Rules**

![AI Guard detection rule](aiguard_detection_rule.png)


---
## 5. Investigate / Signals — Reading a Verdict

Where verdicts land. Each eval shows the `action` (ALLOW/DENY/ABORT), the `attack_categories`, the `blocked` flag,
the sensitivity score, and a **conversation graph**. This is where you confirm the enforcement lab's four scenarios.

**What to look for:**
- Filter to the workshop service + last 15m; find the DENY (scenarios 2–3) and the ABORT (scenario 4).
- The `@ai_guard.blocked:true` flag on a blocked call; the attack-category tags per verdict.
- The conversation graph for the lethal-trifecta ABORT.

**Navigation:** Datadog → Security → AI Guard → **Investigate** (and the **Signals** explorer)

![AI Guard Investigate view](aiguard_investigate.png)

![AI Guard blocked signal](aiguard_blocked_signal.png)


> ⚠️ AI Guard opens a **fresh trace per evaluation** — search by a **±60s timestamp window** + service/env filters,
> not by the agent's Bedrock trace id, or you'll miss the span.

---
## 6. Discover — Agent / Service Protection Map

An inventory of which services/agents are protected by AI Guard and their recent verdict activity — the at-a-glance
view of coverage across your estate.

**What to look for:**
- The two workshop agents appearing as protected services.
- Recent ALLOW/DENY/ABORT counts per service.

**Navigation:** Datadog → Security → AI Guard → **Discover**

![AI Guard Discover map](aiguard_discover.png)

---
## What Gets Captured

| Surface | Field / signal | Where |
|---|---|---|
| APM span | resource `ai_guard`; tags `ai_guard.{target,tool_name,action,reason,blocked}` | Investigate / Traces |
| meta_struct | messages, attack_categories, sds, tag_probs | span detail |
| Security Signal | `@ai_guard.{action,blocked,tools,attack_categories,sds.categories}` | Signals explorer |

**Attack categories (server-side; re-confirm the live list in the console — do not hardcode):** jailbreak,
indirect-prompt-injection, destructive-tool-call, denial-of-service-tool-call, security-exploit, authority-override,
role-play, instruction-override, obfuscation, system-prompt-extraction, data-exfiltration.

## Summary

| Control | Page | Effect | Workshop use |
|---|---|---|---|
| Service Policy | Settings → Service Policies | Monitor vs **Block** (hard gate) | **set to Block** so the hook enforces |
| Sensitivity | Settings → Evaluation Sensitivity | 0–1 flag threshold | tune false-positive rate |
| Tool Blocklist | Settings → Tool Blocklist | hard-deny named tools | optional belt-and-braces |
| Detection Rules | Settings → Detection Rules | signals + automated response | alerting/forensics (not inline gating) |
| Investigate / Signals | Investigate | read verdicts + graph | confirm the enforcement lab |
| Discover | Discover | protection map | coverage at a glance |

**The pluggability lesson:** every behavioural change here is **policy, not code** — flip monitor↔block, retune
sensitivity, or block a tool without redeploying a single agent. That is what makes AI Guard a *pluggable* security
layer in this framework.